# Modelo Supervisado — Predicción de Producción Futura

## Objetivo

El objetivo de esta etapa es construir un dataset de modelado orientado a predecir la producción futura de petróleo de pozos maduros de la Cuenca Cuyana.

A partir de los resultados obtenidos en el EDA, se trabajará únicamente con registros compatibles con producción continua, evitando incorporar períodos asociados a abandono, reparación, estudio, paradas operativas u otras situaciones que puedan distorsionar el comportamiento productivo normal del pozo.

El enfoque seleccionado para la predicción será **directo por horizonte**, donde se entrenarán modelos independientes para predecir producción a 1, 2, 3, 4, 5 y 6 meses hacia adelante.

Antes de construir variables temporales y targets, primero se definirá formalmente el universo de pozos maduros aptos para el modelo.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [2]:
RUTA_DATOS = "../data/processed/produccion_pozos_cuenca_cuyana_2021_2026.csv"

df = pd.read_csv(RUTA_DATOS)

df["fecha_data"] = pd.to_datetime(df["fecha_data"])

df_cuyana = df[df["cuenca"] == "CUYANA"].copy()

print("Filas:", len(df_cuyana))
print("Pozos únicos:", df_cuyana["idpozo"].nunique())
print("Desde:", df_cuyana["fecha_data"].min())
print("Hasta:", df_cuyana["fecha_data"].max())

Filas: 246240
Pozos únicos: 3702
Desde: 2021-01-31 00:00:00
Hasta: 2026-08-31 00:00:00


# 1. Definir dataset de pozos maduros con producción continua

## 1.1 Revisión de estados operativos

In [3]:
df_cuyana["tipoestado"].value_counts(dropna=False)

tipoestado
Abandonado                            75936
Extracción Efectiva                   64195
A Abandonar                           19256
En Inyección Efectiva                 18502
Parado Alta Relación Agua/Petróleo    15451
En Estudio                            14521
En Reserva para Recup. Sec./Asist.    13054
En Espera de Reparación               11720
Parado Transitoriamente               10620
Abandono Temporario                    1968
Otras Situación Inactivo                937
En Reparación                            62
Otras Situación Activo                   18
Name: count, dtype: int64

### Observación

La distribución de `tipoestado` muestra una gran cantidad de registros asociados a situaciones no productivas, como `Abandonado`, `A Abandonar`, `En Estudio`, `En Espera de Reparación` y `Parado Transitoriamente`.

Por este motivo, la presencia de un pozo en el dataset no puede interpretarse automáticamente como actividad productiva.

## 1.2 Relación entre estado operativo y producción de petróleo

In [4]:
estado_prod = (
    df_cuyana
    .groupby("tipoestado")
    .agg(
        registros=("idpozo", "size"),
        pozos=("idpozo", "nunique"),
        prod_pet_media=("prod_pet", "mean"),
        prod_pet_mediana=("prod_pet", "median"),
        pct_prod_pet_positiva=("prod_pet", lambda x: (x > 0).mean() * 100)
    )
    .sort_values("registros", ascending=False)
)

estado_prod.round(2)

,registros,pozos,prod_pet_media,prod_pet_mediana,pct_prod_pet_positiva
tipoestado,,,,,
Abandonado,75936,1340,0.00,0.00,0.00
Extracción Efectiva,64195,1223,84.17,63.57,88.67
A Abandonar,19256,761,0.00,0.00,0.01
En Inyección Efectiva,18502,409,0.01,0.00,0.02
Parado Alta Relación Agua/Petróleo,15451,331,0.00,0.00,0.03
En Estudio,14521,432,0.02,0.00,0.06
En Reserva para Recup. Sec./Asist.,13054,341,0.01,0.00,0.01
En Espera de Reparación,11720,364,0.00,0.00,0.04
Parado Transitoriamente,10620,1017,3.62,0.00,5.46


### Observación

El estado `Extracción Efectiva` se diferencia claramente del resto:

- contiene 64.195 registros;
- comprende 1.223 pozos;
- presenta una producción media aproximada de 84,17 m³;
- una mediana aproximada de 63,57 m³;
- y el 88,67% de sus registros presenta producción de petróleo positiva.

En cambio, la mayoría de los estados asociados a abandono, reparación, estudio o parada presentan una mediana de producción igual a 0 y porcentajes de producción positiva cercanos a 0%.

Por lo tanto, `Extracción Efectiva` se adopta como el estado operativo de referencia para identificar períodos productivos.

## 1.3 Historial productivo general por pozo

In [5]:
historial_pozos = (
    df_cuyana
    .groupby("idpozo")
    .agg(
        meses_totales=("fecha_data", "nunique"),
        meses_prod_positiva=("prod_pet", lambda x: (x > 0).sum()),
        primera_fecha=("fecha_data", "min"),
        ultima_fecha=("fecha_data", "max")
    )
)

historial_pozos["pct_meses_prod_positiva"] = (
    historial_pozos["meses_prod_positiva"] /
    historial_pozos["meses_totales"] * 100
)

historial_pozos.describe()

,meses_totales,meses_prod_positiva,primera_fecha,ultima_fecha,pct_meses_prod_positiva
count,3702.000000,3702.000000,3702,3702,3702.000000
mean,66.515397,15.539708,2021-02-09 17:57:04.959481344,2026-07-26 17:18:11.085899264,23.370405
min,4.000000,0.000000,2021-01-31 00:00:00,2022-04-30 00:00:00,0.000000
25%,67.000000,0.000000,2021-01-31 00:00:00,2026-07-31 00:00:00,0.000000
50%,67.000000,0.000000,2021-01-31 00:00:00,2026-07-31 00:00:00,0.000000
75%,67.000000,26.750000,2021-01-31 00:00:00,2026-07-31 00:00:00,41.791045
max,68.000000,68.000000,2026-05-31 00:00:00,2026-08-31 00:00:00,100.000000
std,4.770967,26.256841,NaN,NaN,39.318476


### Observación

El historial completo muestra que muchos pozos permanecen registrados durante gran parte del período analizado, pero no necesariamente producen petróleo de forma continua.

Aunque la cantidad de meses observados por pozo es alta, la mediana de meses con producción positiva es 0.

Esto confirma que el historial completo no puede utilizarse directamente para definir madurez productiva.

## 1.4 Historial restringido a `Extracción Efectiva`

In [6]:
df_extraccion = df_cuyana[
    df_cuyana["tipoestado"] == "Extracción Efectiva"
].copy()

historial_extraccion = (
    df_extraccion
    .groupby("idpozo")
    .agg(
        meses_extraccion=("fecha_data", "nunique"),
        meses_pet_positiva=("prod_pet", lambda x: (x > 0).sum()),
        prod_pet_media=("prod_pet", "mean"),
        prod_pet_mediana=("prod_pet", "median"),
        primera_fecha=("fecha_data", "min"),
        ultima_fecha=("fecha_data", "max")
    )
)

historial_extraccion["pct_pet_positiva"] = (
    historial_extraccion["meses_pet_positiva"]
    / historial_extraccion["meses_extraccion"]
    * 100
)

historial_extraccion.describe()

,meses_extraccion,meses_pet_positiva,prod_pet_media,prod_pet_mediana,primera_fecha,ultima_fecha,pct_pet_positiva
count,1223.000000,1223.000000,1223.000000,1223.000000,1223,1223,1223.000000
mean,52.489779,46.542927,76.972744,76.689185,2021-04-29 01:14:10.695012096,2025-11-08 01:13:00.049059584,88.072622
min,1.000000,0.000000,0.000000,0.000000,2021-01-31 00:00:00,2021-01-31 00:00:00,0.000000
25%,42.000000,26.500000,26.436954,25.966455,2021-01-31 00:00:00,2025-12-31 00:00:00,98.039216
50%,64.000000,60.000000,60.644375,61.220000,2021-01-31 00:00:00,2026-07-31 00:00:00,100.000000
75%,67.000000,67.000000,105.276543,104.257500,2021-01-31 00:00:00,2026-07-31 00:00:00,100.000000
max,68.000000,68.000000,950.951776,1002.930000,2026-05-31 00:00:00,2026-08-31 00:00:00,100.000000
std,20.163400,24.636266,77.808455,78.549129,NaN,NaN,30.324444


### Observación

Dentro de `Extracción Efectiva`, el comportamiento es mucho más consistente:

- se identificaron 1.223 pozos;
- la mediana es de 64 meses en extracción efectiva;
- la mediana del porcentaje de meses con producción positiva es 100%.

Esto confirma que este subconjunto es más adecuado para definir pozos maduros.

## 1.5 Evaluación de criterios de madurez

In [7]:
for meses in [6, 12, 18, 24, 36]:
    cantidad = (historial_extraccion["meses_extraccion"] >= meses).sum()
    print(f"Pozos con al menos {meses} meses en Extracción Efectiva: {cantidad}")

Pozos con al menos 6 meses en Extracción Efectiva: 1176
Pozos con al menos 12 meses en Extracción Efectiva: 1138
Pozos con al menos 18 meses en Extracción Efectiva: 1089
Pozos con al menos 24 meses en Extracción Efectiva: 1054
Pozos con al menos 36 meses en Extracción Efectiva: 960


### Observación

Incluso utilizando criterios exigentes de historial, el número de pozos disponibles sigue siendo suficiente para modelado.

Resultados:

- al menos 12 meses: 1.138 pozos;
- al menos 18 meses: 1.089 pozos;
- al menos 24 meses: 1.054 pozos;
- al menos 36 meses: 960 pozos.

Esto permite aplicar un criterio de madurez relativamente estricto sin reducir excesivamente la muestra.

In [8]:
for meses in [12, 18, 24]:
    for pct in [80, 90, 95]:
        cantidad = (
            (historial_extraccion["meses_extraccion"] >= meses) &
            (historial_extraccion["pct_pet_positiva"] >= pct)
        ).sum()

        print(
            f">= {meses} meses y >= {pct}% producción positiva: "
            f"{cantidad} pozos"
        )

>= 12 meses y >= 80% producción positiva: 1007 pozos
>= 12 meses y >= 90% producción positiva: 975 pozos
>= 12 meses y >= 95% producción positiva: 920 pozos
>= 18 meses y >= 80% producción positiva: 969 pozos
>= 18 meses y >= 90% producción positiva: 941 pozos
>= 18 meses y >= 95% producción positiva: 894 pozos
>= 24 meses y >= 80% producción positiva: 941 pozos
>= 24 meses y >= 90% producción positiva: 915 pozos
>= 24 meses y >= 95% producción positiva: 868 pozos


### Observación

Al combinar duración de historial y porcentaje de producción positiva, la muestra continúa siendo amplia.

Por ejemplo:

- al menos 24 meses y 90% de producción positiva: 915 pozos;
- al menos 24 meses y 95% de producción positiva: 868 pozos.

Sin embargo, un porcentaje elevado de meses positivos no garantiza que esos meses sean consecutivos.

Por este motivo, se realiza una validación adicional de continuidad temporal.

## 1.6 Verificación de continuidad temporal

In [9]:
df_continuos = df_cuyana[
    (df_cuyana["tipoestado"] == "Extracción Efectiva") &
    (df_cuyana["prod_pet"] > 0)
].copy()

# Representar cada fecha como año-mes
df_continuos["periodo"] = df_continuos["fecha_data"].dt.to_period("M")

# Por seguridad: una fila por pozo-mes
df_continuos = (
    df_continuos
    .sort_values(["idpozo", "periodo"])
    .drop_duplicates(["idpozo", "periodo"])
)

# Número entero del mes para detectar saltos
df_continuos["periodo_num"] = df_continuos["periodo"].astype(int)

# Diferencia respecto al mes anterior del mismo pozo
df_continuos["dif_meses"] = (
    df_continuos.groupby("idpozo")["periodo_num"].diff()
)

# Comienza una nueva racha cuando no viene inmediatamente del mes anterior
df_continuos["nuevo_bloque"] = (
    df_continuos["dif_meses"].ne(1)
)

df_continuos["bloque"] = (
    df_continuos
    .groupby("idpozo")["nuevo_bloque"]
    .cumsum()
)

# Duración de cada racha continua
rachas = (
    df_continuos
    .groupby(["idpozo", "bloque"])
    .size()
)

# Mayor racha continua de cada pozo
max_racha = (
    rachas
    .groupby("idpozo")
    .max()
    .rename("max_meses_continuos")
)

max_racha.describe()

count    1102.000000
mean       43.493648
std        22.196821
min         1.000000
25%        25.000000
50%        45.000000
75%        67.000000
max        68.000000
Name: max_meses_continuos, dtype: float64

### Observación

Las rachas continuas presentan una duración considerable:

- mediana: 45 meses consecutivos;
- percentil 25: 25 meses;
- percentil 75: 67 meses;
- máximo: 68 meses.

Esto indica que existe un conjunto amplio de pozos con trayectorias productivas largas y continuas.

In [10]:
for meses in [6, 12, 18, 24, 36]:
    cantidad = (max_racha >= meses).sum()

    print(
        f"Pozos con al menos {meses} meses consecutivos "
        f"de Extracción Efectiva y producción positiva: {cantidad}"
    )

Pozos con al menos 6 meses consecutivos de Extracción Efectiva y producción positiva: 1047
Pozos con al menos 12 meses consecutivos de Extracción Efectiva y producción positiva: 975
Pozos con al menos 18 meses consecutivos de Extracción Efectiva y producción positiva: 918
Pozos con al menos 24 meses consecutivos de Extracción Efectiva y producción positiva: 851
Pozos con al menos 36 meses consecutivos de Extracción Efectiva y producción positiva: 666


### Observación

La cantidad de pozos con continuidad suficiente es:

- al menos 6 meses consecutivos: 1.047 pozos;
- al menos 12 meses: 975;
- al menos 18 meses: 918;
- al menos 24 meses: 851;
- al menos 36 meses: 666.

Se seleccionó un mínimo de 24 meses consecutivos porque representa dos años de historial productivo continuo y mantiene una muestra amplia de 851 pozos.

In [11]:
pozos_maduros_ids = max_racha[
    max_racha >= 24
].index

df_maduros = df_cuyana[
    df_cuyana["idpozo"].isin(pozos_maduros_ids)
].copy()

print("Pozos maduros:", df_maduros["idpozo"].nunique())
print("Registros totales:", len(df_maduros))

Pozos maduros: 851
Registros totales: 56998


In [12]:
df_modelo_base = df_cuyana[
    df_cuyana["idpozo"].isin(pozos_maduros_ids)
    & (df_cuyana["tipoestado"] == "Extracción Efectiva")
    & (df_cuyana["prod_pet"] > 0)
].copy()

df_modelo_base = (
    df_modelo_base
    .sort_values(["idpozo", "fecha_data"])
    .reset_index(drop=True)
)

print("Pozos:", df_modelo_base["idpozo"].nunique())
print("Registros:", len(df_modelo_base))
print("Desde:", df_modelo_base["fecha_data"].min())
print("Hasta:", df_modelo_base["fecha_data"].max())

Pozos: 851
Registros: 51350
Desde: 2021-01-31 00:00:00
Hasta: 2026-08-31 00:00:00


## 1.7 Selección de la racha productiva principal

In [13]:
tamano_bloques = (
    df_continuos
    .groupby(["idpozo", "bloque"])
    .size()
    .reset_index(name="meses_bloque")
)

bloque_principal = (
    tamano_bloques
    .sort_values(
        ["idpozo", "meses_bloque"],
        ascending=[True, False]
    )
    .drop_duplicates("idpozo")
)

bloque_principal = bloque_principal[
    bloque_principal["meses_bloque"] >= 24
].copy()

bloque_principal.head()

,idpozo,bloque,meses_bloque
11,8380,2,66
25,8385,6,32
27,8386,2,30
32,8387,2,41
36,8388,3,30


### Observación

Para cada pozo se identificó su bloque consecutivo más largo de `Extracción Efectiva` con producción positiva.

Se conservaron únicamente los bloques cuya duración es de al menos 24 meses.

## 1.8 Construcción del dataset final

In [14]:
df_modelo_base = df_continuos.merge(
    bloque_principal[["idpozo", "bloque"]],
    on=["idpozo", "bloque"],
    how="inner"
).copy()

df_modelo_base = (
    df_modelo_base
    .sort_values(["idpozo", "fecha_data"])
    .reset_index(drop=True)
)

print("Pozos:", df_modelo_base["idpozo"].nunique())
print("Registros:", len(df_modelo_base))
print("Desde:", df_modelo_base["fecha_data"].min())
print("Hasta:", df_modelo_base["fecha_data"].max())

Pozos: 851
Registros: 44948
Desde: 2021-01-31 00:00:00
Hasta: 2026-08-31 00:00:00


### Observación

El dataset resultante contiene:

- 851 pozos maduros;
- 44.948 registros;
- información entre enero de 2021 y agosto de 2026.

Al conservar únicamente la racha continua principal de cada pozo, se evita mezclar distintos períodos productivos separados por interrupciones.

## 1.9 Validación final de continuidad

In [15]:
chequeo_continuidad = (
    df_modelo_base
    .groupby("idpozo")["periodo_num"]
    .apply(lambda x: x.sort_values().diff().dropna().eq(1).all())
)

print("Pozos con continuidad perfecta:", chequeo_continuidad.sum())
print("Pozos totales:", len(chequeo_continuidad))
print("Pozos con algún salto:", (~chequeo_continuidad).sum())

Pozos con continuidad perfecta: 851
Pozos totales: 851
Pozos con algún salto: 0


### Observación

La validación confirmó:

- 851 pozos con continuidad perfecta;
- 851 pozos totales;
- 0 pozos con saltos temporales.

Por lo tanto, el dataset seleccionado cumple con el criterio de continuidad necesario para construir variables temporales.

## 1.10 Conclusión final de la seccion 1

Se definió como **pozo maduro con producción continua** a aquel que presenta una racha de al menos **24 meses consecutivos** en estado `Extracción Efectiva`, con producción de petróleo positiva (`prod_pet > 0`) y sin interrupciones temporales dentro de la secuencia seleccionada.

El criterio fue definido a partir del análisis de:

- estado operativo;
- duración del historial;
- porcentaje de producción positiva;
- continuidad mensual real.

Luego de conservar únicamente la racha productiva continua principal de cada pozo, se obtuvo un dataset final compuesto por:

- **851 pozos maduros**;
- **44.948 registros**;
- período entre enero de 2021 y agosto de 2026;
- **0 saltos temporales** dentro de las secuencias seleccionadas.

Este dataset, denominado `df_modelo_base`, será utilizado como base para las siguientes etapas de Feature Engineering del modelo de **predicción directa por horizonte**, incluyendo lags, ventanas móviles, antigüedad productiva y targets futuros H1–H6.

### Guardado del dataset procesado

Se guarda el dataset final de pozos maduros con producción continua en `data/processed/` para utilizarlo como entrada reproducible en las siguientes etapas de Feature Engineering y modelado.

In [16]:
RUTA_MODELO_BASE = "../data/processed/pozos_maduros_extraccion_efectiva_24m.csv"

df_modelo_base.to_csv(
    RUTA_MODELO_BASE,
    index=False
)

print("Archivo guardado en:", RUTA_MODELO_BASE)

Archivo guardado en: ../data/processed/pozos_maduros_extraccion_efectiva_24m.csv


### Observación

El dataset `pozos_maduros_extraccion_efectiva_24m.csv` queda disponible como base procesada para construir lags, ventanas móviles, antigüedad productiva y targets futuros H1–H6.

# 2. Crear variables temporales — Lags

En esta etapa se construyen variables de rezago a partir de la producción histórica de petróleo de cada pozo.

Estas variables permitirán que el modelo utilice información de meses anteriores para realizar la **predicción directa por horizonte** de la producción futura.

In [17]:
RUTA_MODELO_BASE = "../data/processed/pozos_maduros_extraccion_efectiva_24m.csv"

df_fe = pd.read_csv(
    RUTA_MODELO_BASE,
    parse_dates=["fecha_data"]
)

df_fe = (
    df_fe
    .sort_values(["idpozo", "fecha_data"])
    .reset_index(drop=True)
)

print("Pozos:", df_fe["idpozo"].nunique())
print("Registros:", len(df_fe))
print("Desde:", df_fe["fecha_data"].min())
print("Hasta:", df_fe["fecha_data"].max())

Pozos: 851
Registros: 44948
Desde: 2021-01-31 00:00:00
Hasta: 2026-08-31 00:00:00


## 2.1 Preparación del dataset para Feature Engineering

### Observación

Se cargó el dataset procesado de pozos maduros con producción continua y se ordenaron los registros por `idpozo` y `fecha_data`.

El orden cronológico es necesario para construir correctamente las variables de rezago sin mezclar información entre pozos.

In [18]:
df_fe["prod_pet_lag1"] = (
    df_fe.groupby("idpozo")["prod_pet"].shift(1)
)

df_fe["prod_pet_lag2"] = (
    df_fe.groupby("idpozo")["prod_pet"].shift(2)
)

df_fe["prod_pet_lag3"] = (
    df_fe.groupby("idpozo")["prod_pet"].shift(3)
)

df_fe["prod_pet_lag6"] = (
    df_fe.groupby("idpozo")["prod_pet"].shift(6)
)

## 2.2 Construcción de variables de rezago

### Observación

Se generaron cuatro variables temporales de producción:

- `prod_pet_lag1`: producción del mes anterior;
- `prod_pet_lag2`: producción de hace 2 meses;
- `prod_pet_lag3`: producción de hace 3 meses;
- `prod_pet_lag6`: producción de hace 6 meses.

Estas variables representan el comportamiento productivo reciente de cada pozo y serán utilizadas como variables predictoras.

In [19]:
columnas_lags = [
    "idpozo",
    "fecha_data",
    "prod_pet",
    "prod_pet_lag1",
    "prod_pet_lag2",
    "prod_pet_lag3",
    "prod_pet_lag6"
]

df_fe[columnas_lags].head(15)

,idpozo,fecha_data,prod_pet,prod_pet_lag1,prod_pet_lag2,prod_pet_lag3,prod_pet_lag6
0,8380,2021-03-31,15.08,NaN,NaN,NaN,NaN
1,8380,2021-04-30,24.29,15.08,NaN,NaN,NaN
2,8380,2021-05-31,27.10,24.29,15.08,NaN,NaN
3,8380,2021-06-30,25.01,27.10,24.29,15.08,NaN
4,8380,2021-07-31,21.16,25.01,27.10,24.29,NaN
5,8380,2021-08-31,11.16,21.16,25.01,27.10,NaN
6,8380,2021-09-30,10.30,11.16,21.16,25.01,15.08
7,8380,2021-10-31,26.54,10.30,11.16,21.16,24.29
8,8380,2021-11-30,12.03,26.54,10.30,11.16,27.10
9,8380,2021-12-31,10.77,12.03,26.54,10.30,25.01


## 2.3 Verificación visual de los lags

### Observación

La inspección de los primeros registros confirma que las variables de rezago fueron construidas correctamente dentro de cada pozo.

Por ejemplo, para el pozo observado:

- `prod_pet_lag1` corresponde a la producción del mes inmediatamente anterior;
- `prod_pet_lag2` corresponde a la producción de dos meses atrás;
- `prod_pet_lag3` corresponde a la producción de tres meses atrás;
- `prod_pet_lag6` corresponde a la producción de seis meses atrás.

Los valores faltantes de las primeras filas son esperables, ya que al inicio de cada serie todavía no existe suficiente historial previo para calcular todos los rezagos.

In [21]:
columnas_lags = [
    "idpozo",
    "fecha_data",
    "prod_pet",
    "prod_pet_lag1",
    "prod_pet_lag2",
    "prod_pet_lag3",
    "prod_pet_lag6"
]

df_fe[columnas_lags].head(15)

,idpozo,fecha_data,prod_pet,prod_pet_lag1,prod_pet_lag2,prod_pet_lag3,prod_pet_lag6
0,8380,2021-03-31,15.08,NaN,NaN,NaN,NaN
1,8380,2021-04-30,24.29,15.08,NaN,NaN,NaN
2,8380,2021-05-31,27.10,24.29,15.08,NaN,NaN
3,8380,2021-06-30,25.01,27.10,24.29,15.08,NaN
4,8380,2021-07-31,21.16,25.01,27.10,24.29,NaN
5,8380,2021-08-31,11.16,21.16,25.01,27.10,NaN
6,8380,2021-09-30,10.30,11.16,21.16,25.01,15.08
7,8380,2021-10-31,26.54,10.30,11.16,21.16,24.29
8,8380,2021-11-30,12.03,26.54,10.30,11.16,27.10
9,8380,2021-12-31,10.77,12.03,26.54,10.30,25.01


### Observación

La inspección visual confirma que los rezagos fueron construidos correctamente para cada pozo.

En el ejemplo mostrado:

- `prod_pet_lag1` toma la producción del mes anterior;
- `prod_pet_lag2` toma la producción de dos meses atrás;
- `prod_pet_lag3` toma la producción de tres meses atrás;
- `prod_pet_lag6` toma la producción de seis meses atrás.

Los valores `NaN` de las primeras filas son esperables, porque al comienzo de la serie todavía no existe suficiente historial previo para calcular todos los rezagos.

## 2.4 Revisión de valores faltantes generados por los lags

In [22]:
df_fe[
    [
        "prod_pet_lag1",
        "prod_pet_lag2",
        "prod_pet_lag3",
        "prod_pet_lag6"
    ]
].isna().sum()

prod_pet_lag1     851
prod_pet_lag2    1702
prod_pet_lag3    2553
prod_pet_lag6    5106
dtype: int64

### Observación

La cantidad de valores faltantes generados por los lags coincide con lo esperado para 851 pozos:

- `prod_pet_lag1`: 851 valores faltantes;
- `prod_pet_lag2`: 1.702 valores faltantes;
- `prod_pet_lag3`: 2.553 valores faltantes;
- `prod_pet_lag6`: 5.106 valores faltantes.

Estos valores `NaN` no representan errores en los datos, sino la falta natural de historial previo en los primeros meses de cada pozo.

Por ejemplo, para calcular un rezago de 6 meses se necesitan seis observaciones anteriores, por lo que las primeras seis filas de cada pozo no pueden tener un valor válido para `prod_pet_lag6`.

## 2.5 Conclusión de la segunda seccion

Se construyeron correctamente las variables temporales de rezago para la producción de petróleo de cada pozo:

- `prod_pet_lag1`
- `prod_pet_lag2`
- `prod_pet_lag3`
- `prod_pet_lag6`

La validación visual confirmó que cada lag representa correctamente la producción histórica del mismo pozo, respetando el orden cronológico y sin mezclar información entre pozos.

Los valores faltantes observados al inicio de cada serie son esperables y se deben a la ausencia de historial previo suficiente para calcular cada rezago.

Estas variables quedan preparadas para ser utilizadas como predictores en las siguientes etapas de Feature Engineering y, posteriormente, en el modelo de **predicción directa por horizonte H1–H6**.

# 3. Crear ventanas móviles de producción
Las ventanas móviles permiten resumir el comportamiento reciente de la producción de petróleo de cada pozo.

En esta etapa se construyen estadísticas móviles sobre `prod_pet` para capturar nivel promedio y variabilidad reciente antes de realizar la predicción futura.

## 3.1 Construcción de medias móviles

In [24]:
df_fe["prod_pet_roll3_mean"] = (
    df_fe.groupby("idpozo")["prod_pet"]
    .transform(lambda x: x.rolling(window=3, min_periods=3).mean())
)

df_fe["prod_pet_roll6_mean"] = (
    df_fe.groupby("idpozo")["prod_pet"]
    .transform(lambda x: x.rolling(window=6, min_periods=6).mean())
)

print("Medias móviles creadas:")
print(
    df_fe[
        [
            "idpozo",
            "fecha_data",
            "prod_pet",
            "prod_pet_roll3_mean",
            "prod_pet_roll6_mean"
        ]
    ].head(10)
)

Medias móviles creadas:
   idpozo fecha_data  prod_pet  prod_pet_roll3_mean  prod_pet_roll6_mean
0    8380 2021-03-31     15.08                  NaN                  NaN
1    8380 2021-04-30     24.29                  NaN                  NaN
2    8380 2021-05-31     27.10            22.156667                  NaN
3    8380 2021-06-30     25.01            25.466667                  NaN
4    8380 2021-07-31     21.16            24.423333                  NaN
5    8380 2021-08-31     11.16            19.110000            20.633333
6    8380 2021-09-30     10.30            14.206667            19.836667
7    8380 2021-10-31     26.54            16.000000            20.211667
8    8380 2021-11-30     12.03            16.290000            17.700000
9    8380 2021-12-31     10.77            16.446667            15.326667


### Observación

Las medias móviles se generaron correctamente.

La variable `prod_pet_roll3_mean` comienza a tener valores a partir del tercer mes de cada pozo, mientras que `prod_pet_roll6_mean` comienza a partir del sexto mes.

Los valores `NaN` iniciales son esperables, ya que no existe suficiente historial previo para completar cada ventana.

Estas variables permiten resumir el nivel promedio reciente de producción de cada pozo y suavizar fluctuaciones mensuales.

## 3.2 Construcción de desviaciones móviles

In [25]:
df_fe["prod_pet_roll3_std"] = (
    df_fe.groupby("idpozo")["prod_pet"]
    .transform(lambda x: x.rolling(window=3, min_periods=3).std())
)

df_fe["prod_pet_roll6_std"] = (
    df_fe.groupby("idpozo")["prod_pet"]
    .transform(lambda x: x.rolling(window=6, min_periods=6).std())
)

print("Desviaciones móviles creadas:")
print(
    df_fe[
        [
            "idpozo",
            "fecha_data",
            "prod_pet",
            "prod_pet_roll3_std",
            "prod_pet_roll6_std"
        ]
    ].head(10)
)

Desviaciones móviles creadas:
   idpozo fecha_data  prod_pet  prod_pet_roll3_std  prod_pet_roll6_std
0    8380 2021-03-31     15.08                 NaN                 NaN
1    8380 2021-04-30     24.29                 NaN                 NaN
2    8380 2021-05-31     27.10            6.287562                 NaN
3    8380 2021-06-30     25.01            1.459600                 NaN
4    8380 2021-07-31     21.16            3.013143                 NaN
5    8380 2021-08-31     11.16            7.148951            6.248282
6    8380 2021-09-30     10.30            6.037096            7.312118
7    8380 2021-10-31     26.54            9.138030            7.636676
8    8380 2021-11-30     12.03            8.918806            7.392345
9    8380 2021-12-31     10.77            8.763757            6.841437


### Observación

Las desviaciones móviles se calcularon correctamente.

La variable `prod_pet_roll3_std` comienza a tener valores a partir del tercer mes de cada pozo, mientras que `prod_pet_roll6_std` comienza a partir del sexto mes.

Los valores `NaN` iniciales son esperables debido a la falta de observaciones suficientes para completar cada ventana.

Estas variables permiten representar la estabilidad o variabilidad reciente de la producción de cada pozo:

- valores bajos indican una producción relativamente estable;
- valores altos reflejan mayores fluctuaciones en el período reciente.

## 3.3 Verificación conjunta de las ventanas móviles

In [26]:
columnas_rolling = [
    "idpozo",
    "fecha_data",
    "prod_pet",
    "prod_pet_roll3_mean",
    "prod_pet_roll6_mean",
    "prod_pet_roll3_std",
    "prod_pet_roll6_std"
]

print("Verificación conjunta de ventanas móviles:")
print(
    df_fe[columnas_rolling].head(15)
)

Verificación conjunta de ventanas móviles:
    idpozo fecha_data  prod_pet  prod_pet_roll3_mean  prod_pet_roll6_mean  \
0     8380 2021-03-31     15.08                  NaN                  NaN   
1     8380 2021-04-30     24.29                  NaN                  NaN   
2     8380 2021-05-31     27.10            22.156667                  NaN   
3     8380 2021-06-30     25.01            25.466667                  NaN   
4     8380 2021-07-31     21.16            24.423333                  NaN   
5     8380 2021-08-31     11.16            19.110000            20.633333   
6     8380 2021-09-30     10.30            14.206667            19.836667   
7     8380 2021-10-31     26.54            16.000000            20.211667   
8     8380 2021-11-30     12.03            16.290000            17.700000   
9     8380 2021-12-31     10.77            16.446667            15.326667   
10    8380 2022-01-31     10.36            11.053333            13.526667   
11    8380 2022-02-28     10.05  

### Observación

La verificación conjunta confirma que las medias y desviaciones móviles se calcularon correctamente para cada pozo.

Las ventanas de 3 meses reaccionan más rápido a cambios recientes en la producción, mientras que las ventanas de 6 meses presentan un comportamiento más suavizado al considerar un período más amplio.

También se observa que los valores faltantes aparecen únicamente al comienzo de cada serie, hasta que se alcanza la cantidad mínima de observaciones necesaria para completar cada ventana.

## 3.4 Revisión de valores faltantes generados por las ventanas móviles

In [27]:
df_fe[
    [
        "prod_pet_roll3_mean",
        "prod_pet_roll6_mean",
        "prod_pet_roll3_std",
        "prod_pet_roll6_std"
    ]
].isna().sum()

prod_pet_roll3_mean    1702
prod_pet_roll6_mean    4255
prod_pet_roll3_std     1702
prod_pet_roll6_std     4255
dtype: int64

### Observación

Los valores faltantes generados por las ventanas móviles coinciden con lo esperado:

- `prod_pet_roll3_mean`: 1.702 valores faltantes;
- `prod_pet_roll3_std`: 1.702 valores faltantes;
- `prod_pet_roll6_mean`: 4.255 valores faltantes;
- `prod_pet_roll6_std`: 4.255 valores faltantes.

Dado que existen 851 pozos, las ventanas de 3 meses generan valores faltantes en las primeras 2 observaciones de cada pozo, mientras que las ventanas de 6 meses los generan en las primeras 5 observaciones.

Estos valores no representan errores, sino la falta natural de historial suficiente al comienzo de cada secuencia. La eliminación de observaciones sin historial suficiente se realizará posteriormente, una vez construidas todas las variables del modelo.

## 3.5 Conclusión de la seccion 3 

Se construyeron correctamente variables de ventana móvil sobre la producción de petróleo de cada pozo:

- media móvil de 3 meses;
- media móvil de 6 meses;
- desviación estándar móvil de 3 meses;
- desviación estándar móvil de 6 meses.

Estas variables permiten resumir tanto el nivel reciente de producción como su variabilidad en el corto y mediano plazo.

La validación confirmó que los valores faltantes aparecen únicamente en las primeras observaciones de cada pozo, debido a la falta natural de historial suficiente para completar las ventanas.

No se eliminaron estos valores en esta etapa, ya que el filtrado definitivo se realizará una vez construidas todas las variables necesarias para el modelo de **predicción directa por horizonte H1–H6**.

# 4. Crear variable de antigüedad productiva

La antigüedad productiva permite representar cuánto tiempo lleva produciendo cada pozo al momento de cada observación.

Para evitar subestimar la edad de pozos antiguos, se utilizará la fecha real de primera producción disponible en el padrón de pozos, en lugar de tomar únicamente la primera fecha observada en el período 2021–2026.

## 4.1 Revisión del padrón de primera producción

In [28]:
RUTA_PADRON = "../data/raw/02_padrones/padrn-de-pozos-de-captulo-iv-con-fecha-de-primera-produccin.csv"

padron = pd.read_csv(RUTA_PADRON)

print("Dimensiones:", padron.shape)
print("\nColumnas disponibles:")
print(padron.columns.tolist())

padron.head()

Dimensiones: (86197, 3)

Columnas disponibles:
['idpozo', 'anio', 'mes']


,idpozo,anio,mes
0,212,2006,1
1,213,2006,1
2,214,2006,1
3,215,2006,1
4,216,2006,1


## 4.2 Construcción de la fecha de primera producción

In [29]:
padron["fecha_primera_produccion"] = pd.to_datetime(
    dict(
        year=padron["anio"],
        month=padron["mes"],
        day=1
    ),
    errors="coerce"
)

print("Fechas de primera producción construidas:")
print(
    padron[
        ["idpozo", "anio", "mes", "fecha_primera_produccion"]
    ].head(10)
)

print("\nFechas faltantes:")
print(padron["fecha_primera_produccion"].isna().sum())

Fechas de primera producción construidas:
   idpozo  anio  mes fecha_primera_produccion
0     212  2006    1               2006-01-01
1     213  2006    1               2006-01-01
2     214  2006    1               2006-01-01
3     215  2006    1               2006-01-01
4     216  2006    1               2006-01-01
5     217  2006    1               2006-01-01
6     218  2006    1               2006-01-01
7     219  2006    1               2006-01-01
8     220  2006    1               2006-01-01
9     221  2006    1               2006-01-01

Fechas faltantes:
0


### Observación

La fecha de primera producción se construyó correctamente a partir de las columnas `anio` y `mes` del padrón de pozos.

No se detectaron valores faltantes en `fecha_primera_produccion`, por lo que esta variable puede utilizarse para calcular la antigüedad productiva de los pozos.

## 4.3 Incorporación de la fecha de primera producción al dataset de Feature Engineering

In [30]:
df_fe = df_fe.merge(
    padron[["idpozo", "fecha_primera_produccion"]],
    on="idpozo",
    how="left"
)

print("Registros después del merge:", len(df_fe))
print("Pozos:", df_fe["idpozo"].nunique())

print("\nFechas de primera producción faltantes:")
print(df_fe["fecha_primera_produccion"].isna().sum())

print("\nEjemplo:")
print(
    df_fe[
        [
            "idpozo",
            "fecha_data",
            "fecha_primera_produccion"
        ]
    ].head(10)
)

Registros después del merge: 44948
Pozos: 851

Fechas de primera producción faltantes:
0

Ejemplo:
   idpozo fecha_data fecha_primera_produccion
0    8380 2021-03-31               2006-01-01
1    8380 2021-04-30               2006-01-01
2    8380 2021-05-31               2006-01-01
3    8380 2021-06-30               2006-01-01
4    8380 2021-07-31               2006-01-01
5    8380 2021-08-31               2006-01-01
6    8380 2021-09-30               2006-01-01
7    8380 2021-10-31               2006-01-01
8    8380 2021-11-30               2006-01-01
9    8380 2021-12-31               2006-01-01


### Observación

La incorporación de `fecha_primera_produccion` se realizó correctamente.

Después del `merge`, el dataset mantiene:

- **44.948 registros**;
- **851 pozos**;
- **0 fechas de primera producción faltantes**.

Esto confirma que la unión con el padrón no generó duplicados ni pérdidas de información y que todos los pozos del dataset de Feature Engineering cuentan con una fecha válida de primera producción.

## 4.4 Cálculo de la antigüedad productiva en meses

In [31]:
df_fe["antiguedad_productiva_meses"] = (
    (df_fe["fecha_data"].dt.year - df_fe["fecha_primera_produccion"].dt.year) * 12
    + (df_fe["fecha_data"].dt.month - df_fe["fecha_primera_produccion"].dt.month)
)

print("Antigüedad productiva calculada:")
print(
    df_fe[
        [
            "idpozo",
            "fecha_data",
            "fecha_primera_produccion",
            "antiguedad_productiva_meses"
        ]
    ].head(10)
)

print("\nResumen:")
print(df_fe["antiguedad_productiva_meses"].describe())

Antigüedad productiva calculada:
   idpozo fecha_data fecha_primera_produccion  antiguedad_productiva_meses
0    8380 2021-03-31               2006-01-01                          182
1    8380 2021-04-30               2006-01-01                          183
2    8380 2021-05-31               2006-01-01                          184
3    8380 2021-06-30               2006-01-01                          185
4    8380 2021-07-31               2006-01-01                          186
5    8380 2021-08-31               2006-01-01                          187
6    8380 2021-09-30               2006-01-01                          188
7    8380 2021-10-31               2006-01-01                          189
8    8380 2021-11-30               2006-01-01                          190
9    8380 2021-12-31               2006-01-01                          191

Resumen:
count    44948.000000
mean       187.026920
std         54.731815
min          2.000000
25%        184.000000
50%        204.000000


### Observación

La antigüedad productiva se calculó correctamente en meses a partir de la diferencia entre `fecha_data` y `fecha_primera_produccion`.

En el ejemplo del pozo 8380, la antigüedad aumenta de forma consistente en un mes por cada período observado.

El resumen general muestra:

- mínimo: 2 meses;
- mediana: 204 meses;
- media: aproximadamente 187 meses;
- máximo: 247 meses.

Esto indica que el dataset contiene principalmente pozos con una trayectoria productiva extensa, coherente con el objetivo de analizar pozos maduros.

## 4.5 Conclusión de la seccion 4

Se incorporó la fecha real de primera producción de cada pozo a partir del padrón y se construyó la variable `antiguedad_productiva_meses`.

La unión con el padrón mantuvo los **44.948 registros** y los **851 pozos**, sin generar fechas faltantes.

La nueva variable permite representar la edad productiva de cada pozo en cada momento del tiempo y será utilizada como feature en el modelo de **predicción directa por horizonte H1–H6**.

# 5. Construcción de variables objetivo — Horizontes H1–H6

En esta etapa se construyen las variables objetivo que representan la producción futura de petróleo de cada pozo.

Se utiliza un enfoque de **predicción directa por horizonte**, creando un target independiente para cada uno de los próximos seis meses:

- H1: producción a 1 mes;
- H2: producción a 2 meses;
- H3: producción a 3 meses;
- H4: producción a 4 meses;
- H5: producción a 5 meses;
- H6: producción a 6 meses.

Debido a que las secuencias de cada pozo fueron previamente validadas como mensualmente continuas, los desplazamientos temporales representan meses futuros reales.

## 5.1 Construcción de los targets futuros

In [32]:
for h in range(1, 7):
    df_fe[f"target_{h}m"] = (
        df_fe.groupby("idpozo")["prod_pet"].shift(-h)
    )

columnas_targets = [
    "idpozo",
    "fecha_data",
    "prod_pet",
    "target_1m",
    "target_2m",
    "target_3m",
    "target_4m",
    "target_5m",
    "target_6m"
]

print("Targets H1-H6 creados:")
print(df_fe[columnas_targets].head(15))

Targets H1-H6 creados:
    idpozo fecha_data  prod_pet  target_1m  target_2m  target_3m  target_4m  \
0     8380 2021-03-31     15.08      24.29      27.10      25.01      21.16   
1     8380 2021-04-30     24.29      27.10      25.01      21.16      11.16   
2     8380 2021-05-31     27.10      25.01      21.16      11.16      10.30   
3     8380 2021-06-30     25.01      21.16      11.16      10.30      26.54   
4     8380 2021-07-31     21.16      11.16      10.30      26.54      12.03   
5     8380 2021-08-31     11.16      10.30      26.54      12.03      10.77   
6     8380 2021-09-30     10.30      26.54      12.03      10.77      10.36   
7     8380 2021-10-31     26.54      12.03      10.77      10.36      10.05   
8     8380 2021-11-30     12.03      10.77      10.36      10.05      13.03   
9     8380 2021-12-31     10.77      10.36      10.05      13.03      11.82   
10    8380 2022-01-31     10.36      10.05      13.03      11.82      12.38   
11    8380 2022-02-28     10.

### Observación

Los targets futuros H1–H6 se construyeron correctamente para cada pozo.

Cada variable objetivo representa la producción de petróleo del mismo pozo a un horizonte futuro específico:

- `target_1m`: producción a 1 mes;
- `target_2m`: producción a 2 meses;
- `target_3m`: producción a 3 meses;
- `target_4m`: producción a 4 meses;
- `target_5m`: producción a 5 meses;
- `target_6m`: producción a 6 meses.

La inspección visual confirma que los desplazamientos temporales respetan correctamente el orden cronológico de cada pozo.

## 5.2 Revisión de valores faltantes en los targets

In [33]:
df_fe[
    [
        "target_1m",
        "target_2m",
        "target_3m",
        "target_4m",
        "target_5m",
        "target_6m"
    ]
].isna().sum()

target_1m     851
target_2m    1702
target_3m    2553
target_4m    3404
target_5m    4255
target_6m    5106
dtype: int64

### Observación

La cantidad de valores faltantes en los targets futuros coincide con lo esperado para 851 pozos.

Cada horizonte genera valores faltantes en las últimas observaciones de cada serie, ya que no existe información futura suficiente para calcular el target correspondiente.

En particular:

- `target_1m`: 851 valores faltantes;
- `target_2m`: 1.702;
- `target_3m`: 2.553;
- `target_4m`: 3.404;
- `target_5m`: 4.255;
- `target_6m`: 5.106.

Estos valores no representan errores, sino la ausencia natural de observaciones futuras disponibles para cada horizonte.

## 5.3 Definición de filas aptas para modelado

In [34]:
columnas_features = [
    "prod_pet",
    "prod_pet_lag1",
    "prod_pet_lag2",
    "prod_pet_lag3",
    "prod_pet_lag6",
    "prod_pet_roll3_mean",
    "prod_pet_roll6_mean",
    "prod_pet_roll3_std",
    "prod_pet_roll6_std",
    "antiguedad_productiva_meses"
]

columnas_targets = [
    "target_1m",
    "target_2m",
    "target_3m",
    "target_4m",
    "target_5m",
    "target_6m"
]

df_modelado_completo = df_fe.dropna(
    subset=columnas_features + columnas_targets
).copy()

print("Registros originales:", len(df_fe))
print("Registros aptos para H1-H6:", len(df_modelado_completo))
print("Pozos:", df_modelado_completo["idpozo"].nunique())
print("Desde:", df_modelado_completo["fecha_data"].min())
print("Hasta:", df_modelado_completo["fecha_data"].max())

Registros originales: 44948
Registros aptos para H1-H6: 34736
Pozos: 851
Desde: 2021-07-31 00:00:00
Hasta: 2026-02-28 00:00:00


### Observación

Luego de exigir que cada registro tenga disponibles todas las variables predictoras y los seis targets futuros, el dataset apto para modelado quedó compuesto por:

- **34.736 registros**;
- **851 pozos**;
- período entre **julio de 2021 y febrero de 2026**.

La reducción respecto de los 44.948 registros originales es esperable, ya que se excluyen las primeras observaciones de cada pozo que no cuentan con suficiente historial para los lags y las últimas observaciones que no poseen seis meses futuros disponibles para construir `target_6m`.

Este dataset representa el subconjunto histórico completamente utilizable para entrenamiento y backtesting simultáneo de los horizontes H1–H6.

## 5.4 Estrategia temporal de entrenamiento, validación y pronóstico

Para respetar la naturaleza temporal del problema, no se utilizará una división aleatoria de los datos.

La estrategia se organiza en tres etapas:

1. **Entrenamiento histórico:** utilizar observaciones pasadas con features completas y targets conocidos.
2. **Backtesting completo:** evaluar el modelo sobre una fecha de corte anterior que permita verificar H1–H6 con datos reales.
3. **Pronóstico final:** utilizar una fecha de corte reciente, de manera que una parte de los horizontes pueda verificarse y otra represente una predicción futura.

### Evaluación de cobertura temporal para el backtest

In [36]:
cobertura_mensual = (
    df_modelado_completo
    .groupby("fecha_data")
    .agg(
        pozos=("idpozo", "nunique"),
        registros=("idpozo", "size")
    )
    .reset_index()
)

print("Últimos 18 meses disponibles para backtesting:")
print(
    cobertura_mensual
    .tail(18)
    .to_string(index=False)
)

Últimos 18 meses disponibles para backtesting:
fecha_data  pozos  registros
2024-09-30    655        655
2024-10-31    655        655
2024-11-30    653        653
2024-12-31    650        650
2025-01-31    646        646
2025-02-28    634        634
2025-03-31    624        624
2025-04-30    618        618
2025-05-31    599        599
2025-06-30    579        579
2025-07-31    571        571
2025-08-31    556        556
2025-09-30    546        546
2025-10-31    536        536
2025-11-30    530        530
2025-12-31    522        522
2026-01-31    513        513
2026-02-28      7          7


### Observación

La cobertura temporal muestra una disminución progresiva en la cantidad de pozos disponibles hacia el final del período.

Hasta enero de 2026 se mantienen **513 pozos** con información completa para features y targets H1–H6. Sin embargo, en febrero de 2026 la disponibilidad cae abruptamente a solo **7 pozos**.

Por este motivo, febrero de 2026 no se considera suficientemente representativo como fecha principal de backtesting.

Se selecciona **enero de 2026** como fecha de corte para el backtest completo, ya que permite evaluar los seis horizontes hasta julio de 2026 manteniendo una muestra considerable de pozos.

### Backtest completo H1–H6

In [37]:
fecha_backtest = pd.Timestamp("2026-01-31")

df_backtest = df_modelado_completo[
    df_modelado_completo["fecha_data"] == fecha_backtest
].copy()

print("Fecha de backtest:", fecha_backtest.date())
print("Pozos disponibles:", df_backtest["idpozo"].nunique())
print("Registros:", len(df_backtest))

print("\nHorizontes verificables:")
print("H1 -> febrero 2026")
print("H2 -> marzo 2026")
print("H3 -> abril 2026")
print("H4 -> mayo 2026")
print("H5 -> junio 2026")
print("H6 -> julio 2026")

Fecha de backtest: 2026-01-31
Pozos disponibles: 513
Registros: 513

Horizontes verificables:
H1 -> febrero 2026
H2 -> marzo 2026
H3 -> abril 2026
H4 -> mayo 2026
H5 -> junio 2026
H6 -> julio 2026


### Observación

El corte temporal seleccionado en **enero de 2026** permite realizar un backtest completo para los seis horizontes H1–H6.

En esta fecha se dispone de **513 pozos**, una cantidad considerablemente mayor que la obtenida con el corte de febrero de 2026.

Los horizontes verificables son:

- H1: febrero de 2026;
- H2: marzo de 2026;
- H3: abril de 2026;
- H4: mayo de 2026;
- H5: junio de 2026;
- H6: julio de 2026.

Esta fecha se utilizará como backtest principal, ya que combina cercanía temporal con una muestra suficientemente amplia para evaluar el desempeño del modelo.

### Pronóstico reciente con parte verificable y parte futura

In [38]:
fecha_pronostico = pd.Timestamp("2026-05-31")

df_pronostico = df_fe[
    df_fe["fecha_data"] == fecha_pronostico
].copy()

print("Fecha base del pronóstico:", fecha_pronostico.date())
print("Pozos disponibles:", df_pronostico["idpozo"].nunique())
print("Registros:", len(df_pronostico))

print("\nHorizontes:")
print("H1 -> junio 2026      | verificable")
print("H2 -> julio 2026      | verificable")
print("H3 -> agosto 2026     | verificable")
print("H4 -> septiembre 2026 | futuro")
print("H5 -> octubre 2026    | futuro")
print("H6 -> noviembre 2026  | futuro")

Fecha base del pronóstico: 2026-05-31
Pozos disponibles: 530
Registros: 530

Horizontes:
H1 -> junio 2026      | verificable
H2 -> julio 2026      | verificable
H3 -> agosto 2026     | verificable
H4 -> septiembre 2026 | futuro
H5 -> octubre 2026    | futuro
H6 -> noviembre 2026  | futuro


### Observación

La fecha base seleccionada para el pronóstico reciente es **mayo de 2026**.

En esta fecha se dispone de **530 pozos** con información suficiente para construir las variables predictoras.

Los horizontes quedan divididos en dos grupos:

- H1: junio de 2026 — verificable;
- H2: julio de 2026 — verificable;
- H3: agosto de 2026 — verificable;
- H4: septiembre de 2026 — futuro;
- H5: octubre de 2026 — futuro;
- H6: noviembre de 2026 — futuro.

Esta configuración permite evaluar parcialmente el comportamiento del modelo sobre meses ya observados y, al mismo tiempo, generar pronósticos sobre meses todavía no disponibles en el dataset.

In [39]:
RUTA_FE = "../data/processed/pozos_maduros_features_targets_h1_h6.csv"

df_fe.to_csv(
    RUTA_FE,
    index=False
)

print("Dataset guardado en:", RUTA_FE)
print("Registros:", len(df_fe))
print("Pozos:", df_fe["idpozo"].nunique())

Dataset guardado en: ../data/processed/pozos_maduros_features_targets_h1_h6.csv
Registros: 44948
Pozos: 851


## 5.5 Conclusión de la seccion 5

Se construyeron correctamente los targets futuros `target_1m` a `target_6m`, correspondientes a un enfoque de **predicción directa por horizonte**.

También se definieron las observaciones aptas para modelado, exigiendo disponibilidad completa de features y targets, lo que dejó un conjunto histórico de **34.736 registros y 851 pozos** para entrenamiento y backtesting.

Se establecieron dos cortes temporales complementarios:

- **enero de 2026** como fecha de backtest principal, con **513 pozos** y posibilidad de verificar completamente H1–H6 hasta julio de 2026;
- **mayo de 2026** como fecha base del pronóstico reciente, con **530 pozos**, donde H1–H3 pueden verificarse con datos reales y H4–H6 representan predicciones futuras.

Esta estrategia permite evaluar el desempeño del modelo de forma temporalmente consistente y, posteriormente, utilizarlo para generar pronósticos sobre períodos aún no observados.